# 01 — Preprocessing (Figure 1)

**Tujuan.** Bandingkan RGB asli, crop retina, resize, Gaussian blur, V1, dan BG10 pada **contoh train/validation yang dipilih menurut metadata transfer atau guard manifest lokal**. **Metode:** CAD-ViT §III-A, Figure 1, hlm. 91507: crop black border → resize 512×512 (area interpolation dibahas) → Gaussian blur. Threshold 10, kernel 5×5/σ=1, serta EXIF/RGB ialah `ASSUMED`; BG10 ialah `EXPERIMENTAL EXTENSION`. Kotak merah Figure 1 adalah anotasi paper, bukan tensor model.

## Metode dan status bukti

| Tahap | Operasi aktual | Tujuan dan batas interpretasi | Status sumber |
| --- | --- | --- | --- |
| Original RGB | `ImageOps.exif_transpose` lalu RGB | Menjaga orientasi pembacaan; ukuran dan pencahayaan bergantung pada gambar sumber | Panel DDR transfer tersedia; manifest lokal belum ada |
| Crop | Grayscale, mask piksel `>10`, `cv2.boundingRect` | Mengurangi border gelap; ambang 10 adalah **IMPLEMENTATION ASSUMPTION** dan dapat membuang tepi redup | Implementasi V1 terverifikasi dari source |
| Resize | `cv2.INTER_AREA` ke 512×512 | Menyeragamkan input; rasio asli dapat berubah | Ukuran eksplisit paper; interpolasi implementasi |
| Gaussian blur | Kernel 5×5, σx=σy=1 | Menghaluskan variasi lokal, berpotensi mengurangi detail halus | Blur eksplisit paper; kernel/σ asumsi |
| Historical V1 | Hasil blur di atas, `uint8` RGB | `preprocessing_stages` memeriksa kesamaan piksel dengan `preprocess_fundus` | **ACTUAL IMPLEMENTATION** |
| Historical V2 BG10 | V1 lalu `clip(4I−4Gσ10(I)+128,0,255)` | Peningkatan kontras dapat menghasilkan halo/pergeseran warna; bukan preprocessing CAD-ViT asli | **EXPERIMENTAL EXTENSION** |

Paper Figure 1 memperlihatkan contoh sebelum/sesudah crop, resize, dan blur, tetapi tidak menetapkan ambang crop, kernel/σ, atau nilai interpolasi secara lengkap. Sel berikut hanya menerima gambar yang lolos `guarded_non_test_image`: satu baris manifest train/validation, path di dalam data root, dan hash gambar bila tersedia. Panel sesudah eksekusi harus dibaca sebagai **perbandingan metode pada gambar yang sama**, bukan bukti mutu klasifikasi. Manifest dan citra train/validation mentah belum tersedia lokal untuk mengulang guard. Panel DDR statis berikut berasal dari transfer server berhash cocok; identitas split per gambar belum dapat dihitung ulang dari manifest lokal.

[Audit histogram RGB, brightness, contrast, dan kesetiaan preprocessing](preprocessing_analysis.ipynb) adalah pekerjaan lokal terpisah pada citra audit independen. Citra itu belum terhubung ke manifest train/validation APTOS atau DDR, sehingga hasilnya tidak diberi label sampel kampanye.

In [1]:
from pathlib import Path
import os, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "reproduction/cadvit/model.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ARTIFACT_ROOT = Path(os.environ["CAD_VIT_ARTIFACT_ROOT"]).expanduser().resolve() if os.environ.get("CAD_VIT_ARTIFACT_ROOT") else None
print("Artifact root:", ARTIFACT_ROOT if ARTIFACT_ROOT else "PENDING (set CAD_VIT_ARTIFACT_ROOT)")

Artifact root: PENDING (set CAD_VIT_ARTIFACT_ROOT)


In [2]:
from reproduction.cadvit.visualization import guarded_non_test_image, preprocessing_stages, plot_preprocessing
image_path = os.environ.get("CAD_VIT_AUDIT_IMAGE")
manifest_path = os.environ.get("CAD_VIT_AUDIT_MANIFEST")
data_root = os.environ.get("CAD_VIT_DATA_ROOT")
source_split = os.environ.get("CAD_VIT_AUDIT_SPLIT", "train")
if image_path and manifest_path and data_root:
    try:
        from PIL import Image
        checked_path, row = guarded_non_test_image(image_path, data_root, manifest_path, source_split)
        with Image.open(checked_path) as image:
            stages = preprocessing_stages(image)
        print("VERIFIED", row["image_id"], row["split"], "crop xywh", stages["crop_xywh"])
        try:
            fig = plot_preprocessing(stages)
            from IPython.display import display
            display(fig)
        except ImportError as exc:
            print("PENDING plot dependency:", exc)
    except (ImportError, FileNotFoundError, ValueError) as exc:
        print("UNRESOLVED:", exc)
else:
    print("PENDING: set CAD_VIT_AUDIT_IMAGE, CAD_VIT_AUDIT_MANIFEST, CAD_VIT_DATA_ROOT (train/val only)")

PENDING: set CAD_VIT_AUDIT_IMAGE, CAD_VIT_AUDIT_MANIFEST, CAD_VIT_DATA_ROOT (train/val only)


## Citra DDR aktual: train, preprocessing, dan BG10

Panel ini berasal dari transfer visualisasi server yang SHA-256 ZIP-nya cocok dengan catatan peneliti; [daftar hash tiap PNG dan lisensi](../report_figures/README.md#ddr-real-image-visualization-transfer). Label **train** dan grade tercetak pada gambar server, sedangkan manifest per gambar belum tersedia lokal untuk verifikasi ulang. Gambar memperlihatkan variasi luas border hitam, orientasi diskus optik, pencahayaan, dan kejernihan; perbedaan tersebut adalah pengamatan visual, bukan penilaian lesi teranotasi.

![DDR: satu citra train berlabel grade 0–4](../report_figures/DDR_train_grade_gallery.png)

Setiap panel berikut memakai **gambar sumber yang sama** pada enam tahap: original → crop → resize 512×512 → Gaussian blur → V1 → V2 BG10. V1 terlihat hampir sama dengan panel blur karena memang keluaran tahap itu; BG10 adalah ekstensi eksperimental, bukan langkah eksplisit CAD-ViT.

| Grade / ID pada gambar | Pengamatan langsung dari enam panel |
| --- | --- |
| 0 · `20170428161917714.jpg` | Crop membuang border hitam luas dan memperbesar cakram fundus; BG10 menonjolkan pembuluh dan menghasilkan halo biru pada tepi. |
| 1 · `007-1843-100.jpg` | Fundus sudah hampir memenuhi bingkai, sehingga crop kecil; blur V1 halus, sedangkan BG10 mengubah latar menjadi abu kebiruan dan menonjolkan tepi. |
| 2 · `20170608112856878.jpg` | Resize memperbesar bidang retina; area terang dan pembuluh tampak lebih kontras pada BG10, bersama perubahan warna yang nyata. |
| 3 · `007-4564-200.jpg` | Original relatif gelap; crop dan resize menaikkan ukuran tampilan, sedangkan BG10 menaikkan kontras pembuluh sekaligus pola tepi. |
| 4 · `007-6904-400.jpg` | Original berkontras rendah dengan beberapa bercak terang; BG10 menonjolkan bercak dan lingkaran/halo, tanpa membuktikan bahwa detail itu diagnostik. |

![DDR grade 0: enam tahap pada citra yang sama](../report_figures/DDR_preprocessing_grade0.png)
![DDR grade 1: enam tahap pada citra yang sama](../report_figures/DDR_preprocessing_grade1.png)
![DDR grade 2: enam tahap pada citra yang sama](../report_figures/DDR_preprocessing_grade2.png)
![DDR grade 3: enam tahap pada citra yang sama](../report_figures/DDR_preprocessing_grade3.png)
![DDR grade 4: enam tahap pada citra yang sama](../report_figures/DDR_preprocessing_grade4.png)

## Citra APTOS2019 aktual — akses lokal bagi pemegang hak dataset

Kesembilan PNG APTOS dan dua JSON metadata disalin byte demi byte dari arsip transfer yang [dicatat di `report_figures`](../report_figures/README.md#aptos2019-local-only-visualization-transfer). File APTOS diabaikan Git karena izin berbagi kepada kolaborator belum terkonfirmasi. Panel Markdown ini terlihat **tanpa kernel pada salinan lokal yang memiliki file tersebut**; tautannya tidak akan menampilkan gambar pada GitHub atau clone baru. Jangan menyalin panel ke artefak yang diterbitkan sebelum hak berbagi dikonfirmasi.

![APTOS2019 lokal: satu citra train berlabel grade 0–4](../report_figures/APTOS2019_train_grade_gallery.png)

Lima gambar berikut masing-masing menampilkan original → crop → resize 512×512 → Gaussian blur → V1 → V2 BG10 pada gambar sumber yang sama. Label grade berasal dari panel transfer; manifest/gambar sumber tidak tersedia lokal untuk pencocokan ulang. BG10 adalah ekstensi eksperimental, bukan preprocessing kampanye V1.

| Grade | Pengamatan langsung pada panel preprocessing APTOS |
| --- | --- |
| 0 | Fundus berbentuk lingkaran dengan border hitam luas; crop mengurangi border, dan BG10 memberi pergeseran warna serta halo di tepi. |
| 1 | Bingkai asli lebih dekat dan gelap; perubahan V1 halus, sedangkan BG10 meningkatkan kontras tepi dan mengubah warna. |
| 2 | Pola pembuluh dan titik terang terlihat; BG10 memperjelas sebagian detail sekaligus memperkuat halo/warna yang tidak ada pada RGB asli. |
| 3 | Panel asli lebih terang kemerahan; crop/resize mengubah skala tampilan, dan BG10 menggeser warna ke abu kebiruan. |
| 4 | Detail fundus relatif gelap dengan area gelap di satu sisi; BG10 mengubah kontras tetapi tidak membuktikan informasi diagnostik bertambah. |

![APTOS2019 lokal grade 0: enam tahap pada citra yang sama](../report_figures/APTOS2019_preprocessing_grade0.png)
![APTOS2019 lokal grade 1: enam tahap pada citra yang sama](../report_figures/APTOS2019_preprocessing_grade1.png)
![APTOS2019 lokal grade 2: enam tahap pada citra yang sama](../report_figures/APTOS2019_preprocessing_grade2.png)
![APTOS2019 lokal grade 3: enam tahap pada citra yang sama](../report_figures/APTOS2019_preprocessing_grade3.png)
![APTOS2019 lokal grade 4: enam tahap pada citra yang sama](../report_figures/APTOS2019_preprocessing_grade4.png)

Variasi border, pencahayaan, dan warna adalah observasi visual; paket ini tidak memuat anotasi lesi untuk menguji dampak tahap preprocessing pada penilaian klinis.

## Augmentasi train dan penyeimbangan

Urutan runner modular adalah `dataset_for` → `FundusSplit.__getitem__` → `Image.open` → `preprocess_fundus` (array RGB `uint8`) → jika split `train` dan `augment_train=True`, `RandomHorizontalFlip(p=0.5)` → `RandomVerticalFlip(p=0.5)` → `RandomAffine(degrees=15, shear=10, interpolation=BILINEAR, fill=0)` → `torch.from_numpy(...).permute(2,0,1).float()/255`. Validation/test tidak memakai transform acak. `RandomAffine` mengambil rotasi dan shear dari rentang konfigurasi, sehingga satu output tidak mewakili seluruh kemungkinan. Border hitam dari `fill=0` dan interpolasi bilinear dapat mengubah detail tepi. [Source runner](../cadvit/experiments.py) adalah rujukan parameter yang berlaku untuk campaign modular; runner V1/V2 historis pada [arsip](../archive/CAD_ViT_original_105_cells.ipynb) harus dibaca menurut versi dan run masing-masing.

Preprocessing deterministik mengubah setiap gambar sebelum model. Augmentasi geometri menghasilkan variasi saat gambar train diambil. Oversampling mengulangi **pemilihan ID gambar** kelas minoritas, bukan membuat gambar sumber unik; variasi augmentasi pada draw ulang dapat berbeda. Validation dan final test tidak di-oversample. Output pipeline gabungan DDR dari server tersedia di bawah; contoh transform individual belum dikirim. Untuk visualisasi individual kelak, catat seed, ID, grade, split, dimensi, hash, dan parameter transform untuk tiap panel. Jangan menafsirkan demo deterministik itu sebagai transform yang tepat terjadi pada epoch historis.

## Output augmentasi DDR yang tersimpan

![DDR: V1 dan tiga keluaran pipeline augmentasi modular per grade](../report_figures/DDR_augmentation_gallery.png)

Lima baris memakai gambar train grade 0–4 yang sama dengan galeri di atas. Kolom kiri adalah V1 tanpa augmentasi; tiga kolom lain adalah keluaran **pipeline gabungan** dengan seed 42, 43, dan 44, bukan demonstrasi masing-masing transform secara terpisah. Beberapa keluaran seed 42 tampak sangat dekat dengan masukan; seed 43 memperlihatkan pembalikan sisi diskus dan rotasi pada beberapa baris; segitiga hitam di tepi tampak saat affine memakai `fill=0`. Seed 44 kembali mengubah orientasi/bingkai pada sebagian gambar. Karena probabilitas dan parameter affine diambil acak, gambar ini tidak boleh dianggap transform yang persis terjadi pada epoch training tertentu. Jumlah gambar sumber unik tetap lima pada galeri DDR ini.

## Output augmentasi APTOS2019 — lokal saja

![APTOS2019 lokal: V1 dan tiga keluaran augmentasi gabungan per grade](../report_figures/APTOS2019_augmentation_gallery.png)

Panel APTOS juga memakai satu gambar train per grade 0–4, dengan V1 di kiri dan output pipeline gabungan untuk seed 42, 43, dan 44. Beberapa panel seed 42 mirip dengan V1; pada seed lain tampak pembalikan/orientasi berbeda dan tepi segitiga hitam akibat affine `fill=0`. Ini menunjukkan variasi geometri pada contoh terpilih, bukan transform yang pasti terjadi pada epoch training tertentu. Oversampling dapat mengambil ID yang sama berkali-kali; galeri ini tidak menunjukkan bertambahnya citra sumber unik. File APTOS hanya tersedia pada salinan lokal yang memiliki hak akses dataset; status berbagi dijelaskan pada [catatan artefak](../report_figures/README.md#aptos2019-local-only-visualization-transfer).

**Interpretasi dan keterbatasan.** Sel kode guarded di atas hanya menampilkan hasil bila path dan keanggotaan split lolos pemeriksaan. Panel statis DDR dan APTOS yang ditransfer ditampilkan terpisah di notebook ini; panel APTOS hanya tersedia lokal dan dikecualikan dari Git. Visual yang menyerupai Figure 1 tidak membuktikan parameter penulis identik; BG10 tidak dipakai pada kampanye reproduksi V1. Periksa lesi kecil dan halo warna sebelum menyimpulkan efek visual. Notebook audit lama `preprocessing_analysis.ipynb` tetap dipertahankan sebagai referensi historis.